# Predictive Maintenance — AI4I 2020

Goal: Predict machine failure with high precision and useful recall to reduce unplanned downtime.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
import os, pathlib

# Paths
DATA_DIR = pathlib.Path('data')
ASSETS_DIR = pathlib.Path('assets'); ASSETS_DIR.mkdir(exist_ok=True, parents=True)
PLOTS_DIR = pathlib.Path('plots'); PLOTS_DIR.mkdir(exist_ok=True, parents=True)


In [ ]:
# Load data (place ai4i2020.csv in the data/ folder)
csv_path = DATA_DIR / 'ai4i2020.csv'
assert csv_path.exists(), f"Missing {csv_path}. Download from UCI and place it in data/."
df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
# Quick EDA
display(df.describe(include='all').T)
print("Nulls by column:\n", df.isna().sum())
# Example label(s): 'Machine failure' or 'Target' depending on distro variation
candidate_labels = [c for c in df.columns if 'fail' in c.lower() or c.lower()=='target']
print("Candidate label columns:", candidate_labels)


In [ ]:
# --- Adjust to match your file's column names ---
# Common variants: 'Machine failure' (binary), or 'Target'
LABEL = 'Machine failure' if 'Machine failure' in df.columns else ('Target' if 'Target' in df.columns else candidate_labels[0])
FEATURES = [c for c in df.columns if c != LABEL and df[c].dtype != 'O']  # numeric features only for baseline

X = df[FEATURES].values
y = df[LABEL].astype(int).values
print(LABEL, len(FEATURES), "features")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Baseline: logistic regression with class weights
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logit = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
cal_logit = CalibratedClassifierCV(logit, cv=3, method='sigmoid')
cal_logit.fit(X_train_s, y_train)
proba = cal_logit.predict_proba(X_test_s)[:,1]

ap = average_precision_score(y_test, proba)
roc = roc_auc_score(y_test, proba)
print(f"Baseline (Calibrated Logistic) — PR-AUC: {ap:.3f}, ROC-AUC: {roc:.3f}")


In [ ]:
# Tree-based model
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample', n_jobs=-1)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:,1]
ap_rf = average_precision_score(y_test, proba_rf)
roc_rf = roc_auc_score(y_test, proba_rf)
print(f"RandomForest — PR-AUC: {ap_rf:.3f}, ROC-AUC: {roc_rf:.3f}")


In [ ]:
def recall_at_precision(y_true, y_proba, precision_target=0.90):
    p, r, th = precision_recall_curve(y_true, y_proba)
    # Find first index where precision >= target
    idx = next((i for i, v in enumerate(p) if v >= precision_target), None)
    if idx is None:
        return 0.0, 1.0  # recall=0, threshold=1 (degenerate)
    return float(r[idx]), float(th[idx-1]) if idx>0 else float(th[0])

for name, yhat in [('Logistic', proba), ('RandomForest', proba_rf)]:
    rec, thr = recall_at_precision(y_test, yhat, precision_target=0.90)
    print(f"{name}: Recall @ 90% precision = {rec:.3f} at threshold {thr:.3f}")


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

disp = PrecisionRecallDisplay.from_predictions(y_test, proba_rf)
plt.title("RandomForest Precision-Recall")
plt.savefig(PLOTS_DIR / "pr_curve_rf.png", dpi=160, bbox_inches='tight')
plt.show()


### Next
- Add SHAP/feature importances
- Try time-aware split if you derive a temporal index
- Convert operating point to avoided-downtime economics